# 🏦 Phase 2 OPTIMISÉE — Attijari Bank 
### Détection d'opportunités + Enrichissement Elasticsearch
---


## CELLULE 0 — Installation des dépendances

In [1]:
import subprocess
import sys

print('📦 Installation des dépendances...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 
                       'elasticsearch', 'pandas', 'numpy', 'scikit-learn',
                       '-q'])
print('✅ Installation terminée ')

📦 Installation des dépendances...


✅ Installation terminée 


## CELLULE 1 — Imports et Configuration

In [2]:
import pandas as pd
import numpy as np
import json
import warnings
import os
import logging
import time
from datetime import datetime
from elasticsearch import Elasticsearch, helpers
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder
from collections import defaultdict
import subprocess
import sys

warnings.filterwarnings('ignore')

# Configuration logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print('✅ Imports terminés')

C:\Users\Amal Brinsi\AppData\Local\Temp\ipykernel_6136\3387869296.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


✅ Imports terminés


## CELLULE 2 — Configuration Elasticsearch

In [3]:
# Configuration Elasticsearch
ES_HOST = os.environ.get('ES_HOST', 'http://localhost:9200')
INDEX_ORIGINAL = 'attijari-logs'
INDEX_ENRICHED = 'attijari-logs-enriched'

# Timestamp d'exécution
timestamp_phase2 = datetime.now().isoformat()

# Connexion Elasticsearch
es = None
try:
    es = Elasticsearch(ES_HOST)
    if es.ping():
        print(f'✅ Elasticsearch connecté : {ES_HOST}')
    else:
        raise Exception('Ping failed')
except Exception as e:
    print(f'⚠️  Elasticsearch non disponible : {e}')
    print('   Index ne sera pas créé, fichier JSON seulement')
    es = None

print(f'🕐 Timestamp Phase 2 : {timestamp_phase2}')

2026-05-11 21:58:11,224 - INFO - HEAD http://localhost:9200/ [status:200 duration:0.033s]


✅ Elasticsearch connecté : http://localhost:9200
🕐 Timestamp Phase 2 : 2026-05-11T21:58:11.190462


## CELLULE 3 — Charger et préparer les données

In [4]:
print('⏳ Chargement du CSV...')
start_time = time.time()

df = pd.read_csv('attijari_bank_logs_10000.csv', sep=';')

# Prétraitement
df['timestamp']        = pd.to_datetime(df['timestamp'])
df['success']          = df['success'].astype(str).str.strip().map({'True': True, 'False': False})
df['duree_action_sec'] = pd.to_numeric(df['duree_action_sec'], errors='coerce').fillna(0.0)
df['error_code']       = df['error_code'].fillna('NONE')
df['field_modified']   = df['field_modified'].fillna('NONE')
df['heure']            = df['timestamp'].dt.hour
df['jour_semaine']     = df['timestamp'].dt.dayofweek
df['mois']             = df['timestamp'].dt.month

load_time = time.time() - start_time

print('='*70)
print('  PHASE 2 OPTIMISÉE — Informations générales')
print('='*70)
print(f'  Logs chargés       : {len(df):,}')
print(f'  Période            : {df["timestamp"].min().date()} → {df["timestamp"].max().date()}')
print(f'  Taux d\'échec       : {(~df["success"]).mean()*100:.1f}%')
print(f'  Utilisateurs       : {df["user_id"].nunique()}')
print(f'  Agences            : {df["agence"].nunique()}')
print(f'  Applications       : {df["application"].nunique()}')
print(f'  Processus          : {df["processus"].nunique()}')
print(f'  Actions distinctes : {df["action"].nunique()}')
print(f'\n  ⏱️  Temps chargement : {load_time:.2f}s')

⏳ Chargement du CSV...
  PHASE 2 OPTIMISÉE — Informations générales
  Logs chargés       : 10,000
  Période            : 2025-01-01 → 2026-02-05
  Taux d'échec       : 25.2%
  Utilisateurs       : 80
  Agences            : 10
  Applications       : 4
  Processus          : 10


  Actions distinctes : 12

  ⏱️  Temps chargement : 0.09s


## CELLULE 4 — Clustering MÉTIER (sans NLP)

In [5]:
print('\n⏳ Clustering métier ')
start_time = time.time()

# Grouper par (action, processus, application)
# → Plus pertinent que clustering sémantique
df['cluster_key'] = df['action'] + '|' + df['processus'] + '|' + df['application']
df['cluster'] = pd.factorize(df['cluster_key'])[0]

clustering_time = time.time() - start_time

print(f'\n✅ {df["cluster"].nunique()} clusters métier créés')
print(f'⏱️  Temps clustering : {clustering_time:.3f}s')

# Analyse des clusters
print('\n' + '='*75)
print('   Clusters métier — (action | processus | application)')
print('='*75)

cluster_stats = []
for c in sorted(df['cluster'].unique()):
    s = df[df['cluster'] == c]
    nb = len(s)
    taux_echec = round((~s['success']).mean() * 100, 1)
    duree_moy = round(s['duree_action_sec'].mean(), 1)
    action = s['action'].iloc[0]
    processus = s['processus'].iloc[0]
    application = s['application'].iloc[0]
    
    cluster_stats.append({
        'Cluster': c,
        'Logs': nb,
        'Taux échec (%)': taux_echec,
        'Durée moy (s)': duree_moy,
        'Action': action,
        'Processus': processus,
        'Application': application
    })

df_clusters = pd.DataFrame(cluster_stats).set_index('Cluster')
print(df_clusters.head(20).to_string())
if len(df_clusters) > 20:
    print(f'\n... et {len(df_clusters)-20} autres clusters')
print(f'\n✅ Clustering terminé')


⏳ Clustering métier 



✅ 480 clusters métier créés
⏱️  Temps clustering : 0.019s

   Clusters métier — (action | processus | application)


         Logs  Taux échec (%)  Durée moy (s)            Action                     Processus                Application
Cluster                                                                                                                
0          17             5.9            8.7             LOGIN          DEMANDE_CREDIT_CONSO               Attijari Net
1          18             5.6            6.0           APPROVE  OUVERTURE_COMPTE_PARTICULIER            Credit Platform
2          23             0.0            6.8            REJECT              DEMANDE_CHEQUIER  Mobile Banking Backoffice
3          21             0.0            9.1            REJECT  OUVERTURE_COMPTE_PARTICULIER            Credit Platform
4          14             0.0            5.7        FILL_FIELD         PAIEMENT_FACTURE_STEG            Credit Platform
5          18             0.0            6.0         OPEN_FORM              VIREMENT_INTERNE           Core Banking Web
6          22            95.5           

## CELLULE 5 — Détection Répétitions

In [6]:
print('\n⏳ Détection répétitions...')
start_time = time.time()

repetitions = []

# R001 : SESSION_EXPIRED répétées par user/processus
for action in ['SESSION_EXPIRED', 'ERROR_VALIDATION']:
    g = df[df['action'] == action].groupby(['user_id', 'processus']).size().reset_index(name='nb')
    for _, r in g[g['nb'] >= 3].iterrows():
        repetitions.append({
            'type'              : f'REPETITION_{action}',
            'user_id'           : r['user_id'],
            'processus'         : r['processus'],
            'nb_occurrences'    : int(r['nb']),
            'impact'            : 'HAUT' if r['nb'] >= 5 else 'MOYEN',
            'cout_estime_tnd'   : round(r['nb'] * 2 * 0.5, 2),
        })

# R003 : FILL_FIELD répétés
fill = df[df['action'] == 'FILL_FIELD'].groupby(['user_id', 'processus', 'field_modified']).size().reset_index(name='nb')
for _, r in fill[fill['nb'] >= 4].iterrows():
    repetitions.append({
        'type'              : 'REPETITION_FILL_FIELD',
        'user_id'           : r['user_id'],
        'processus'         : r['processus'],
        'field'             : r['field_modified'],
        'nb_occurrences'    : int(r['nb']),
        'impact'            : 'MOYEN',
        'cout_estime_tnd'   : round(r['nb'] * 1.5 * 0.5, 2),
    })

rep_time = time.time() - start_time

print(f'\n✅ {len(repetitions)} répétitions détectées')
print(f'⏱️  Temps détection : {rep_time:.3f}s')
print('\n  Top 5 :')
for r in sorted(repetitions, key=lambda x: x['nb_occurrences'], reverse=True)[:5]:
    print(f"    {r['type']:35s} | {r['user_id']} | {r['processus']:25s} | {r['nb_occurrences']}x")


⏳ Détection répétitions...

✅ 146 répétitions détectées
⏱️  Temps détection : 0.085s

  Top 5 :
    REPETITION_ERROR_VALIDATION         | USER0022 | VIREMENT_INTERNATIONAL    | 6x
    REPETITION_ERROR_VALIDATION         | USER0002 | OPPOSITION_CARTE          | 5x
    REPETITION_ERROR_VALIDATION         | USER0004 | PAIEMENT_FACTURE_SONED    | 5x
    REPETITION_ERROR_VALIDATION         | USER0079 | OPPOSITION_CARTE          | 5x
    REPETITION_SESSION_EXPIRED          | USER0003 | VIREMENT_INTERNE          | 4x


## CELLULE 6 — Erreurs fréquentes

In [7]:
print('\n⏳ Détection erreurs fréquentes...')
start_time = time.time()

descriptions = {
    'ERR_001': 'Timeout connexion',
    'ERR_101': 'Erreur validation formulaire',
    'ERR_201': 'Session expirée',
    'ERR_301': 'Données manquantes',
    'ERR_401': 'Accès non autorisé',
    'ERR_501': 'Erreur serveur interne',
    'ERR_601': 'Données dupliquées',
    'ERR_701': 'Erreur réseau',
    'ERR_901': 'Erreur permission'
}

cout_erreur = {
    'ERR_001': 3.0, 'ERR_101': 1.5, 'ERR_201': 2.0, 'ERR_301': 1.0,
    'ERR_401': 5.0, 'ERR_501': 8.0, 'ERR_601': 2.5, 'ERR_701': 1.5, 'ERR_901': 4.0
}

erreurs = []
grp = (df[df['error_code'] != 'NONE']
       .groupby(['error_code', 'processus', 'application'])
       .size().reset_index(name='nb')
       .query('nb >= 5')
       .sort_values('nb', ascending=False))

for _, r in grp.iterrows():
    cout = cout_erreur.get(r['error_code'], 2.0) * r['nb']
    erreurs.append({
        'type'              : 'ERREUR_FREQUENTE',
        'error_code'        : r['error_code'],
        'description'       : descriptions.get(r['error_code'], 'Inconnue'),
        'processus'         : r['processus'],
        'application'       : r['application'],
        'nb_occurrences'    : int(r['nb']),
        'impact'            : 'CRITIQUE' if r['nb']>=20 else 'HAUT' if r['nb']>=10 else 'MOYEN',
        'cout_estime_tnd'   : round(cout, 2),
    })

err_time = time.time() - start_time

print(f'\n✅ {len(erreurs)} erreurs fréquentes détectées')
print(f'⏱️  Temps détection : {err_time:.3f}s')
print('\n  Top 5 :')
for e in sorted(erreurs, key=lambda x: x['nb_occurrences'], reverse=True)[:5]:
    print(f"    {e['error_code']} | {e['processus']:28s} | {e['nb_occurrences']:3d}x | {e['impact']:8s} | {e['cout_estime_tnd']} TND")


⏳ Détection erreurs fréquentes...



✅ 304 erreurs fréquentes détectées
⏱️  Temps détection : 0.064s

  Top 5 :
    ERR_301 | PAIEMENT_FACTURE_SONED       |  15x | HAUT     | 15.0 TND
    ERR_201 | DEMANDE_CREDIT_CONSO         |  14x | HAUT     | 28.0 TND
    ERR_901 | VIREMENT_INTERNATIONAL       |  13x | HAUT     | 52.0 TND
    ERR_501 | DEMANDE_CREDIT_CONSO         |  13x | HAUT     | 104.0 TND
    ERR_201 | DEMANDE_CHEQUIER             |  13x | HAUT     | 26.0 TND


## CELLULE 7 — Goulots d'étranglement

In [8]:
print('\n⏳ Détection goulots d\'étranglement...')
start_time = time.time()

# Calculer moyenne + écart-type par (processus, action)
stats = df.groupby(['processus', 'action'])['duree_action_sec'].agg(['mean', 'std', 'count']).reset_index()
stats['seuil'] = stats['mean'] + 2 * stats['std'].fillna(0)

# Identifier logs anormalement lents
df_lents = df.merge(stats, on=['processus', 'action'])
df_lents = df_lents[df_lents['duree_action_sec'] > df_lents['seuil']]

# Agréger goulots
goulots = []
grp_g = (df_lents.groupby(['processus', 'action'])
         .agg(nb=('duree_action_sec', 'count'),
              moy=('duree_action_sec', 'mean'),
              mx=('duree_action_sec', 'max'))
         .reset_index()
         .query('nb >= 3')
         .sort_values('moy', ascending=False))

for _, r in grp_g.iterrows():
    goulots.append({
        'type'              : 'GOULOT_ETRANGLEMENT',
        'processus'         : r['processus'],
        'action'            : r['action'],
        'nb_occurrences'    : int(r['nb']),
        'duree_moy_sec'     : round(float(r['moy']), 2),
        'duree_max_sec'     : round(float(r['mx']), 2),
        'impact'            : 'HAUT' if r['moy'] > 15 else 'MOYEN',
        'cout_estime_tnd'   : round(r['nb'] * 2, 2),
    })

goulot_time = time.time() - start_time

print(f'\n✅ {len(goulots)} goulots détectés')
print(f'⏱️  Temps détection : {goulot_time:.3f}s')
print('\n  Top 5 :')
for g in goulots[:5]:
    print(f"    {g['action']:20s} | {g['processus']:25s} | moy: {g['duree_moy_sec']:5.1f}s | {g['cout_estime_tnd']} TND")


⏳ Détection goulots d'étranglement...

✅ 112 goulots détectés
⏱️  Temps détection : 0.055s

  Top 5 :
    SUBMIT_FORM          | OPPOSITION_CARTE          | moy:  44.1s | 6 TND
    SESSION_EXPIRED      | VIREMENT_INTERNATIONAL    | moy:  39.4s | 6 TND
    SESSION_EXPIRED      | DEMANDE_CREDIT_CONSO      | moy:  38.4s | 6 TND
    UPLOAD_DOCUMENT      | CHANGE_DEVISES            | moy:  38.0s | 8 TND
    LOGOUT               | DEMANDE_CARTE_VISA        | moy:  37.1s | 8 TND


## CELLULE 8 — Isolation Forest (Anomalies ML) ← LA VRAIE VALEUR

In [9]:
print('\n⏳ Isolation Forest (détection anomalies ML)...')
start_time = time.time()

# Préparer features pour Isolation Forest
df_ml = df.copy()

# Encoder les champs catégoriels
for col in ['action', 'processus', 'application', 'agence']:
    df_ml[col + '_enc'] = LabelEncoder().fit_transform(df_ml[col])

df_ml['suc'] = df_ml['success'].astype(int)

# Features pour détection anomalies
feats = ['action_enc', 'processus_enc', 'application_enc', 'agence_enc',
         'duree_action_sec', 'suc', 'heure', 'jour_semaine']

# Isolation Forest
iso = IsolationForest(contamination=0.05, random_state=42, n_estimators=100)
df_ml['anom'] = iso.fit_predict(df_ml[feats].values)
df_ml['score'] = iso.decision_function(df_ml[feats].values)

# Extraire anomalies
df_anom = df_ml[df_ml['anom'] == -1]
anomalies = []

for _, r in df_anom.iterrows():
    anomalies.append({
        'type'              : 'ANOMALIE_COMPORTEMENTALE',
        'user_id'           : r['user_id'],
        'agence'            : r['agence'],
        'processus'         : r['processus'],
        'action'            : r['action'],
        'duree_sec'         : round(float(r['duree_action_sec']), 2),
        'success'           : bool(r['success']),
        'error_code'        : r['error_code'],
        'score_anomalie'    : round(float(r['score']), 4),
        'impact'            : 'CRITIQUE' if r['score'] < -0.2 else 'HAUT',
        'cout_estime_tnd'   : round(abs(float(r['score'])) * 10, 2),
    })

iso_time = time.time() - start_time

print(f'\n✅ {len(anomalies)} anomalies détectées')
print(f'  ├─ CRITIQUES : {len([a for a in anomalies if a["impact"]=="CRITIQUE"])}')
print(f'  └─ HAUTES    : {len([a for a in anomalies if a["impact"]=="HAUT"])}')
print(f'⏱️  Temps détection : {iso_time:.3f}s')
print('\n  Top 5 utilisateurs anormaux :')
for user, nb in pd.Series([a['user_id'] for a in anomalies]).value_counts().head(5).items():
    print(f"    {user} → {nb} anomalies")


⏳ Isolation Forest (détection anomalies ML)...


✅ 500 anomalies détectées
  ├─ CRITIQUES : 0
  └─ HAUTES    : 500
⏱️  Temps détection : 0.993s

  Top 5 utilisateurs anormaux :
    USER0058 → 11 anomalies
    USER0050 → 11 anomalies
    USER0008 → 11 anomalies
    USER0073 → 11 anomalies
    USER0077 → 11 anomalies


## CELLULE 9 — Scoring des opportunités

In [10]:
print('\n⏳ Scoring opportunités RPA...')
start_time = time.time()

def score_priorite(frequence, temps_min, cout_tnd, poids_freq=0.3, poids_temps=0.3, poids_cout=0.4):
    """Score composite 0-100 pour prioriser opportunités"""
    score_freq = min(frequence / 50, 1.0) * 100
    score_temps = min(temps_min / 30, 1.0) * 100
    score_cout = min(cout_tnd / 100, 1.0) * 100
    return round(poids_freq*score_freq + poids_temps*score_temps + poids_cout*score_cout, 1)

opportunites = []

# Opportunités à partir des répétitions
for r in repetitions:
    freq = r['nb_occurrences']
    temps = freq * 2
    cout = r.get('cout_estime_tnd', freq * 1)
    score = score_priorite(freq, temps, cout)
    
    if freq >= 5:
        opportunites.append({
            'categorie'       : 'AUTOMATISATION_RPA',
            'priorite'        : 1 if score >= 60 else 2,
            'score_composite' : score,
            'titre'           : f"Reconnexion auto {r['user_id']} sur {r['processus']}",
            'robot_rpa'       : 'RobotReconnexionAuto',
            'nb_occurrences'  : freq,
            'gain_temps_min'  : temps,
            'gain_cout_tnd'   : cout,
        })

# Opportunités à partir des erreurs
for e in erreurs:
    freq = e['nb_occurrences']
    temps = freq * 3
    cout = e.get('cout_estime_tnd', freq * 2)
    score = score_priorite(freq, temps, cout)
    
    if freq >= 15:
        opportunites.append({
            'categorie'       : 'CORRECTION_PROACTIVE',
            'priorite'        : 1 if score >= 60 else 2,
            'score_composite' : score,
            'titre'           : f"Auto-correction {e['error_code']} sur {e['processus']}",
            'robot_rpa'       : f"RobotCorrection_{e['error_code']}",
            'nb_occurrences'  : freq,
            'gain_temps_min'  : temps,
            'gain_cout_tnd'   : cout,
        })

# Opportunités à partir des goulots
for g in goulots:
    freq = g['nb_occurrences']
    temps = freq * g['duree_moy_sec'] / 60
    cout = temps * 0.5
    score = score_priorite(freq, temps, cout)
    
    if g['duree_moy_sec'] > 12:
        opportunites.append({
            'categorie'       : 'OPTIMISATION_PROCESSUS',
            'priorite'        : 2 if score >= 40 else 3,
            'score_composite' : score,
            'titre'           : f"Optimiser {g['action']} sur {g['processus']}",
            'robot_rpa'       : f"RobotOptimise_{g['action']}",
            'nb_occurrences'  : freq,
            'gain_temps_min'  : round(temps, 1),
            'gain_cout_tnd'   : round(cout, 2),
        })

opportunites.sort(key=lambda x: x['score_composite'], reverse=True)

opp_time = time.time() - start_time

print(f'\n✅ {len(opportunites)} opportunités RPA identifiées')
print(f'⏱️  Temps scoring : {opp_time:.3f}s')
print('\n  Top 5 :')
for o in opportunites[:5]:
    print(f"    [{o['score_composite']:5.1f}] {o['titre']:50s} | {o['robot_rpa']}")


⏳ Scoring opportunités RPA...

✅ 117 opportunités RPA identifiées
⏱️  Temps scoring : 0.002s

  Top 5 :
    [ 45.0] Auto-correction ERR_301 sur PAIEMENT_FACTURE_SONED | RobotCorrection_ERR_301
    [ 18.0] Reconnexion auto USER0022 sur VIREMENT_INTERNATIONAL | RobotReconnexionAuto
    [ 15.0] Reconnexion auto USER0002 sur OPPOSITION_CARTE     | RobotReconnexionAuto
    [ 15.0] Reconnexion auto USER0004 sur PAIEMENT_FACTURE_SONED | RobotReconnexionAuto
    [ 15.0] Reconnexion auto USER0079 sur OPPOSITION_CARTE     | RobotReconnexionAuto


## CELLULE 10 — Créer & enrichir index Elasticsearch

In [11]:
index_time = 0

if es is None:
    logger.error('❌ Elasticsearch non disponible. Skipping ES indexation.')
    print('⚠️  Index Elasticsearch ne sera pas créé')
else:
    print('\n⏳ Création index enrichi dans Elasticsearch...')
    start_time = time.time()
    
    try:
        # Supprimer ancien index enrichi
        if es.indices.exists(index=INDEX_ENRICHED):
            es.indices.delete(index=INDEX_ENRICHED)
            logger.info(f"Index '{INDEX_ENRICHED}' supprimé.")
        
        # Créer nouvel index avec mapping complet
        es.indices.create(index=INDEX_ENRICHED, mappings={
            "properties": {
                # Champs originaux
                "timestamp"         : {"type": "date"},
                "user_id"           : {"type": "keyword"},
                "agence"            : {"type": "keyword"},
                "application"       : {"type": "keyword"},
                "processus"         : {"type": "keyword"},
                "action"            : {"type": "keyword"},
                "duree_action_sec"  : {"type": "float"},
                "success"           : {"type": "boolean"},
                "error_code"        : {"type": "keyword"},
                "field_modified"    : {"type": "keyword"},
                "screen"            : {"type": "keyword"},
                "heure"             : {"type": "integer"},
                "jour_semaine"      : {"type": "integer"},
                "mois"              : {"type": "integer"},
                
                # ✨ CHAMPS ENRICHIS PHASE 2
                "cluster"           : {"type": "integer"},
                "score_anomalie"    : {"type": "float"},
                "impact"            : {"type": "keyword"},
                "opportunity_type"  : {"type": "keyword"},
                "robot_suggere"     : {"type": "keyword"},
                "gain_estime_min"   : {"type": "float"},
                "phase2_timestamp"  : {"type": "date"},
                "is_anomalie"       : {"type": "boolean"},
                "is_erreur"         : {"type": "boolean"},
                "is_goulot"         : {"type": "boolean"},
                "is_repetition"     : {"type": "boolean"},
            }
        })
        logger.info(f"Index '{INDEX_ENRICHED}' créé avec mapping.")
        
        # Créer mapping enrichissements
        enrichissement_map = {}
        
        # Anomalies
        for anom in anomalies:
            user_id = anom['user_id']
            processus = anom['processus']
            action = anom['action']
            key = (user_id, processus, action)
            enrichissement_map[key] = {
                'score_anomalie'   : anom['score_anomalie'],
                'impact'           : anom['impact'],
                'is_anomalie'      : True,
                'robot_suggere'    : 'RobotAlerteAnomalie' if anom['impact']=='CRITIQUE' else 'RobotDetectionAnomalie'
            }
        
        # Répétitions
        for rep in repetitions:
            user_id = rep['user_id']
            processus = rep['processus']
            logs_match = df[(df['user_id']==user_id) & (df['processus']==processus)]
            for _, log in logs_match.iterrows():
                key = (log['user_id'], log['processus'], log['action'])
                if key not in enrichissement_map:
                    enrichissement_map[key] = {}
                enrichissement_map[key].update({
                    'is_repetition'    : True,
                    'opportunity_type' : 'AUTOMATISATION_RPA',
                    'robot_suggere'    : 'RobotReconnexionAuto',
                    'gain_estime_min'  : rep['nb_occurrences'] * 2
                })
        
        # Erreurs
        for err in erreurs:
            error_code = err['error_code']
            logs_match = df[df['error_code']==error_code]
            for _, log in logs_match.iterrows():
                key = (log['user_id'], log['processus'], log['action'])
                if key not in enrichissement_map:
                    enrichissement_map[key] = {}
                enrichissement_map[key].update({
                    'is_erreur'        : True,
                    'opportunity_type' : 'CORRECTION_PROACTIVE',
                    'robot_suggere'    : f"RobotCorrection_{error_code}",
                    'gain_estime_min'  : err.get('cout_estime_tnd', 0) / 0.5,
                    'impact'           : err['impact']
                })
        
        # Goulots
        for goulot in goulots:
            processus = goulot['processus']
            action = goulot['action']
            logs_match = df[(df['processus']==processus) & (df['action']==action)]
            for _, log in logs_match.iterrows():
                key = (log['user_id'], log['processus'], log['action'])
                if key not in enrichissement_map:
                    enrichissement_map[key] = {}
                enrichissement_map[key].update({
                    'is_goulot'        : True,
                    'opportunity_type' : 'OPTIMISATION_PROCESSUS',
                    'robot_suggere'    : f"RobotOptimise_{action}",
                    'gain_estime_min'  : goulot['nb_occurrences'] * goulot['duree_moy_sec'] / 60 * 0.4,
                    'impact'           : goulot['impact']
                })
        
        # Indexer docs enrichis
        docs_enrichis = []
        for idx, row in df.iterrows():
            key = (row['user_id'], row['processus'], row['action'])
            enriched_doc = row.to_dict()
            enriched_doc['timestamp'] = row['timestamp'].isoformat()
            enriched_doc['phase2_timestamp'] = timestamp_phase2
            enriched_doc['is_anomalie'] = False
            enriched_doc['is_erreur'] = False
            enriched_doc['is_goulot'] = False
            enriched_doc['is_repetition'] = False
            
            if key in enrichissement_map:
                enriched_doc.update(enrichissement_map[key])
            
            docs_enrichis.append({
                "_index"  : INDEX_ENRICHED,
                "_id"     : f"{idx}",
                "_source" : enriched_doc
            })
        
        # Bulk indexation
        print('   ⏳ Bulk indexation (500 docs/batch)...')
        ok, errors = helpers.bulk(es, docs_enrichis, raise_on_error=False, chunk_size=500)
        index_time = time.time() - start_time
        logger.info(f"✅ {ok} logs enrichis indexés dans '{INDEX_ENRICHED}' en {index_time:.2f}s")
        
        if errors:
            logger.warning(f"⚠️ {len(errors)} erreurs indexation")
            for err in errors[:5]:
                logger.warning(f"   {err}")
        
        print(f"\n✅ {ok} documents indexés en {index_time:.2f}s")
        
    except Exception as e:
        logger.error(f"❌ Erreur indexation : {e}")
        print(f"❌ Erreur : {e}")

2026-05-11 21:58:13,933 - INFO - HEAD http://localhost:9200/attijari-logs-enriched [status:404 duration:0.004s]



⏳ Création index enrichi dans Elasticsearch...


2026-05-11 21:58:14,445 - INFO - PUT http://localhost:9200/attijari-logs-enriched [status:200 duration:0.513s]


2026-05-11 21:58:14,448 - INFO - Index 'attijari-logs-enriched' créé avec mapping.


   ⏳ Bulk indexation (500 docs/batch)...


...

2026-05-11 21:58:29,981 - INFO - PUT http://localhost:9200/_bulk [status:200 duration:0.034s]


2026-05-11 21:58:30,072 - INFO - PUT http://localhost:9200/_bulk [status:200 duration:0.079s]


2026-05-11 21:58:30,117 - INFO - PUT http://localhost:9200/_bulk [status:200 duration:0.033s]


2026-05-11 21:58:30,119 - INFO - ✅ 10000 logs enrichis indexés dans 'attijari-logs-enriched' en 16.19s



✅ 10000 documents indexés en 16.19s


## CELLULE 11 — Export JSON + Rapport Final

In [12]:
print('\n⏳ Export résultats...')
start_time = time.time()

# Export JSON
resultats = {
    'timestamp'        : timestamp_phase2,
    'elasticsearch'    : {
        'status'       : 'INDEXED' if es is not None else 'NOT_AVAILABLE',
        'index'        : INDEX_ENRICHED,
        'host'         : ES_HOST
    },
    'statistics'       : {
        'total_logs'   : len(df),
        'taux_echec'   : round((~df['success']).mean()*100, 2),
        'clusters'     : int(df['cluster'].nunique())
    },
    'repetitions'      : repetitions,
    'erreurs'          : erreurs,
    'goulots'          : goulots,
    'anomalies'        : anomalies,
    'opportunites'     : opportunites
}

with open('resultats_phase2.json', 'w', encoding='utf-8') as f:
    json.dump(resultats, f, ensure_ascii=False, indent=2)

export_time = time.time() - start_time

# Rapport Final
gain_total = sum(o.get('gain_temps_min', 0) for o in opportunites)
cout_total = sum(o.get('gain_cout_tnd', 0) for o in opportunites)

print(f'✅ resultats_phase2.json exporté en {export_time:.3f}s')

total_time = (load_time + clustering_time + rep_time + err_time + goulot_time + 
              iso_time + opp_time + index_time + export_time)

rapport = f"""
{'='*70}
  RAPPORT FINAL — Phase 2 OPTIMISÉE (SANS NLP)
  Attijari Bank | {timestamp_phase2}
{'='*70}

📊 DONNÉES ANALYSÉES
   Logs totaux          : {len(df):,}
   Période              : {df['timestamp'].min().date()} → {df['timestamp'].max().date()}
   Taux d'échec         : {(~df['success']).mean()*100:.1f}%
   Utilisateurs         : {df['user_id'].nunique()}
   Agences              : {df['agence'].nunique()}
   Processus            : {df['processus'].nunique()}

🔍 PROBLÈMES DÉTECTÉS
   Répétitions          : {len(repetitions)}
   Erreurs fréquentes   : {len(erreurs)}
   Goulots              : {len(goulots)}
   Anomalies ML         : {len(anomalies)} (Isolation Forest 5%)
   Opportunités RPA     : {len(opportunites)}

🚀 OPPORTUNITÉS DÉTECTÉES
   AUTOMATISATION       : {len([o for o in opportunites if o['categorie']=='AUTOMATISATION_RPA'])}
   CORRECTION_PROACTIVE : {len([o for o in opportunites if o['categorie']=='CORRECTION_PROACTIVE'])}
   OPTIMISATION         : {len([o for o in opportunites if o['categorie']=='OPTIMISATION_PROCESSUS'])}

💰 GAIN ESTIMÉ
   Temps économisé      : {gain_total:.0f} minutes ({gain_total/60:.1f} heures)
   Coût économisé       : {cout_total:.2f} TND
   ROI annuel projeté   : {cout_total*12:.0f} TND

🗄️ ELASTICSEARCH
   Status               : {resultats['elasticsearch']['status']}
   Index enrichi        : {INDEX_ENRICHED}
   Docs indexés         : {len(df):,}

⏱️  PERFORMANCE (SANS NLP)
   Chargement CSV       : {load_time:.3f}s
   Clustering métier    : {clustering_time:.3f}s
   Répétitions          : {rep_time:.3f}s
   Erreurs              : {err_time:.3f}s
   Goulots              : {goulot_time:.3f}s
   Isolation Forest     : {iso_time:.3f}s ✨ LA VRAIE VALEUR
   Scoring              : {opp_time:.3f}s
   Indexation ES        : {index_time:.3f}s
   Export               : {export_time:.3f}s
   ────────────────────────
   TOTAL                : {total_time:.2f}s ({total_time/60:.2f}m)

✅ Phase 2 OPTIMISÉE terminée !
   → NLP supprimé (gain ~162s)
   → Isolation Forest conservé (détecte anomalies)
   → Prochaine étape : Phase 3 (n8n RPA)
{'='*70}
"""

print(rapport)
logger.info(rapport)

2026-05-11 21:58:30,169 - INFO - 
  RAPPORT FINAL — Phase 2 OPTIMISÉE (SANS NLP)
  Attijari Bank | 2026-05-11T21:58:11.190462

📊 DONNÉES ANALYSÉES
   Logs totaux          : 10,000
   Période              : 2025-01-01 → 2026-02-05
   Taux d'échec         : 25.2%
   Utilisateurs         : 80
   Agences              : 10
   Processus            : 10

🔍 PROBLÈMES DÉTECTÉS
   Répétitions          : 146
   Erreurs fréquentes   : 304
   Goulots              : 112
   Anomalies ML         : 500 (Isolation Forest 5%)
   Opportunités RPA     : 117

🚀 OPPORTUNITÉS DÉTECTÉES
   AUTOMATISATION       : 4
   CORRECTION_PROACTIVE : 1
   OPTIMISATION         : 112

💰 GAIN ESTIMÉ
   Temps économisé      : 299 minutes (5.0 heures)
   Coût économisé       : 142.30 TND
   ROI annuel projeté   : 1708 TND

🗄️ ELASTICSEARCH
   Status               : INDEXED
   Index enrichi        : attijari-logs-enriched
   Docs indexés         : 10,000

⏱️  PERFORMANCE (SANS NLP)
   Chargement CSV       : 0.094s
   Clusterin


⏳ Export résultats...
✅ resultats_phase2.json exporté en 0.017s

  RAPPORT FINAL — Phase 2 OPTIMISÉE (SANS NLP)
  Attijari Bank | 2026-05-11T21:58:11.190462

📊 DONNÉES ANALYSÉES
   Logs totaux          : 10,000
   Période              : 2025-01-01 → 2026-02-05
   Taux d'échec         : 25.2%
   Utilisateurs         : 80
   Agences              : 10
   Processus            : 10

🔍 PROBLÈMES DÉTECTÉS
   Répétitions          : 146
   Erreurs fréquentes   : 304
   Goulots              : 112
   Anomalies ML         : 500 (Isolation Forest 5%)
   Opportunités RPA     : 117

🚀 OPPORTUNITÉS DÉTECTÉES
   AUTOMATISATION       : 4
   CORRECTION_PROACTIVE : 1
   OPTIMISATION         : 112

💰 GAIN ESTIMÉ
   Temps économisé      : 299 minutes (5.0 heures)
   Coût économisé       : 142.30 TND
   ROI annuel projeté   : 1708 TND

🗄️ ELASTICSEARCH
   Status               : INDEXED
   Index enrichi        : attijari-logs-enriched
   Docs indexés         : 10,000

⏱️  PERFORMANCE (SANS NLP)
   Chargement

## CELLULE 12 — Vérification finale (Optionnel)

In [13]:
# Vérification optionnelle
print('\n' + '='*70)
print('  VÉRIFICATION FINALE')
print('='*70)

if es is not None:
    try:
        # Compter docs dans ES
        count = es.count(index=INDEX_ENRICHED)['count']
        print(f'✅ Elasticsearch : {count} docs indexés')
        
        # Compter anomalies
        anom_count = es.count(index=INDEX_ENRICHED, query={'term': {'is_anomalie': True}})['count']
        print(f'✅ Anomalies : {anom_count}')
        
        # Compter erreurs
        err_count = es.count(index=INDEX_ENRICHED, query={'term': {'is_erreur': True}})['count']
        print(f'✅ Erreurs : {err_count}')
        
        # Compter goulots
        goulot_count = es.count(index=INDEX_ENRICHED, query={'term': {'is_goulot': True}})['count']
        print(f'✅ Goulots : {goulot_count}')
    except Exception as e:
        print(f'⚠️  Erreur vérification : {e}')
else:
    print('⚠️  Elasticsearch non disponible')

# Vérifier JSON
import os
if os.path.exists('resultats_phase2.json'):
    size = os.path.getsize('resultats_phase2.json') / 1024 / 1024
    print(f'✅ JSON : resultats_phase2.json ({size:.2f} MB)')
else:
    print('❌ JSON : fichier non trouvé')



2026-05-11 21:58:30,212 - INFO - POST http://localhost:9200/attijari-logs-enriched/_count [status:200 duration:0.012s]


2026-05-11 21:58:30,265 - INFO - POST http://localhost:9200/attijari-logs-enriched/_count [status:200 duration:0.049s]


2026-05-11 21:58:30,316 - INFO - POST http://localhost:9200/attijari-logs-enriched/_count [status:200 duration:0.051s]



  VÉRIFICATION FINALE
✅ Elasticsearch : 3500 docs indexés
✅ Anomalies : 305


✅ Erreurs : 2748


2026-05-11 21:58:30,369 - INFO - POST http://localhost:9200/attijari-logs-enriched/_count [status:200 duration:0.050s]


✅ Goulots : 9388
✅ JSON : resultats_phase2.json (0.36 MB)
